# TODO Solutions — Self-Supervised Representations

This notebook contains complete implementations for all TODO tasks.

| # | Task | Points |
|---|------|---------|
| 1 | `UniversalSlicer` with multiple cut/pad policies | 2+ pts |
| 2 | Rewrite `Audio2SpecPad` (remove hacky pad) | 1 pt |
| 3 | `Wav2Vec2FeatureExtractor` accepting batched `torch.Tensor` | 1 pt |
| 4 | What does `Wav2Vec2ForCTC` predict? | theory |
| 5 | Use `attention_mask` with `Wav2Vec2Model` | 0.5 pt |
| 6 | Normalize `HubertForCTC` outputs to W2V format | 2 pts |
| 7 | Add batching to `XVectorExtractor` + Wav2Vec2→XVector | 1.5 pts |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import numpy as np
from tqdm import tqdm
from transformers import (
    AutoFeatureExtractor, AutoProcessor, Wav2Vec2Model,
    HubertForCTC, AutoModelForAudioXVector, Wav2Vec2FeatureExtractor
)

---
## TODO 1 — `UniversalSlicer` (2+ points)

Implements multiple cut/pad policies:
- `right_pad` — pad silence on the right (default, most common)
- `left_pad` — pad silence on the left
- `center_pad` — pad silence symmetrically on both sides
- `cut` — truncate to `target_length` (from the start)
- `cut_silence` — remove leading/trailing silence using energy threshold, then pad/cut

Returns: `(batched_audio [B, L], padding_mask [B, L])`

In [ ]:
class UniversalSlicer:
    """
    Cuts or pads a batch of 1-D waveforms to a fixed target_length.

    Policies
    --------
    'right_pad'     : zero-pad on the right  (most common for SSL models)
    'left_pad'      : zero-pad on the left
    'center_pad'    : zero-pad symmetrically on both sides
    'cut'           : truncate to target_length from the start
    'cut_silence'   : strip leading/trailing silence (via energy threshold),
                      then apply the chosen pad policy on the remainder

    Parameters
    ----------
    target_length   : int   — desired output length in samples
    pad_policy      : str   — one of the policies above
    silence_db      : float — dB threshold used by 'cut_silence'
    """

    POLICIES = {"right_pad", "left_pad", "center_pad", "cut", "cut_silence"}

    def __init__(
        self,
        target_length: int,
        pad_policy: str = "right_pad",
        silence_db: float = -40.0,
    ):
        assert pad_policy in self.POLICIES, (
            f"Unknown policy '{pad_policy}'. Choose from {self.POLICIES}"
        )
        self.target_length = target_length
        self.pad_policy    = pad_policy
        self.silence_db    = silence_db

    # ------------------------------------------------------------------
    # public API
    # ------------------------------------------------------------------

    def __call__(
        self,
        waves: list,          # list of 1-D np.ndarray or torch.Tensor
    ) -> tuple:
        """Returns (batched_audio [B, L], padding_mask [B, L])."""
        processed = [self._process_single(w) for w in waves]

        batched = torch.zeros(len(processed), self.target_length)
        mask    = torch.zeros(len(processed), self.target_length)

        for i, wave in enumerate(processed):
            length = min(len(wave), self.target_length)   # safety clamp
            batched[i, :length] = wave[:length]
            mask[i,    :length] = 1.0

        return batched, mask

    # ------------------------------------------------------------------
    # internal helpers
    # ------------------------------------------------------------------

    def _to_tensor(self, wave) -> torch.Tensor:
        if isinstance(wave, np.ndarray):
            return torch.from_numpy(wave.astype(np.float32))
        return wave.float()

    def _process_single(self, wave) -> torch.Tensor:
        wave = self._to_tensor(wave)
        L    = self.target_length

        if self.pad_policy == "cut_silence":
            wave = self._strip_silence(wave)
            # after stripping, fall through to right_pad
            return self._pad(wave, L, mode="right_pad")

        if self.pad_policy == "cut" or len(wave) >= L:
            return wave[:L]   # truncate (no padding needed if already long enough)

        return self._pad(wave, L, mode=self.pad_policy)

    @staticmethod
    def _pad(wave: torch.Tensor, L: int, mode: str) -> torch.Tensor:
        """Pad `wave` to exactly `L` samples according to `mode`."""
        n = len(wave)
        if n >= L:
            return wave[:L]
        needed = L - n

        if mode == "right_pad":
            return F.pad(wave, (0, needed))          # pad at the end

        if mode == "left_pad":
            return F.pad(wave, (needed, 0))          # pad at the start

        if mode == "center_pad":
            left  = needed // 2
            right = needed - left
            return F.pad(wave, (left, right))

        raise ValueError(f"Unknown pad mode: {mode}")

    def _strip_silence(self, wave: torch.Tensor) -> torch.Tensor:
        """
        Remove leading and trailing silence.
        A frame is considered 'silent' when its RMS power is below self.silence_db.
        Uses a small 512-sample frame with 256-sample hop.
        """
        frame_size = 512
        hop        = 256

        # Compute RMS per frame
        frames = wave.unfold(0, frame_size, hop)          # [N_frames, frame_size]
        rms_db = 20 * torch.log10(frames.pow(2).mean(dim=1).sqrt().clamp(min=1e-9))

        active = rms_db > self.silence_db                 # [N_frames] bool mask

        if not active.any():
            return wave   # fully silent — return as-is

        first_active = active.nonzero(as_tuple=True)[0][0].item()
        last_active  = active.nonzero(as_tuple=True)[0][-1].item()

        start = first_active * hop
        end   = min(last_active  * hop + frame_size, len(wave))

        return wave[start:end]

In [ ]:
# ── Quick sanity test ──────────────────────────────────────────────────
dummy_waves = [
    np.random.randn(12_000).astype(np.float32),  # shorter than target
    np.random.randn(32_000).astype(np.float32),  # longer  than target
    np.random.randn(16_000).astype(np.float32),  # exact
]

for policy in ["right_pad", "left_pad", "center_pad", "cut", "cut_silence"]:
    slicer = UniversalSlicer(target_length=16_000, pad_policy=policy)
    audio, mask = slicer(dummy_waves)
    assert audio.shape == (3, 16_000), f"Shape mismatch for policy={policy}"
    assert mask.shape  == (3, 16_000)
    print(f"[{policy:12s}]  audio={tuple(audio.shape)}  "
          f"active_frames={mask.sum(dim=1).int().tolist()}")

---
## TODO 2 — Rewrite `Audio2SpecPad` without the hacky pad (1 point)

**The original problem:** The original class uses `nn.MaxPool1d` to *approximate* the audio→spectrogram time reduction, then manually prepends a chunk of ones to account for the mismatch between the pooled length and the actual spectrogram time dimension. That extra `torch.cat` is the "hacky pad".

**The fix:** The correct approach is to compute the expected spectrogram time length analytically using the same formula the spectrogram layer uses: `T_spec = floor((L_audio - kernel_size) / hop_size) + 1`. We then build the mask directly by repeating each audio-mask frame `hop_size` times and cropping/padding to exactly `T_spec`. No MaxPool, no manual cat.

In [ ]:
class Audio2SpecPad(nn.Module):
    """
    Maps an audio-level padding mask (shape [B, L_audio]) to a
    spectrogram-level padding mask (shape [B, T_spec]).

    The mapping follows the standard STFT formula:
        T_spec = floor((L_audio - n_fft) / hop_size) + 1

    Instead of using MaxPool (which only approximates this) we compute
    the mask analytically:
      1. Count how many *audio* samples are non-padded per batch item.
      2. Apply the STFT formula to get the number of valid *spec frames*.
      3. Build a binary mask of shape [B, T_spec].

    Parameters
    ----------
    hop_size : int  — STFT hop length (must match the spectrogram layer)
    n_fft    : int  — FFT window size  (must match the spectrogram layer)
    """

    def __init__(self, hop_size: int = 512, n_fft: int = 1024):
        super().__init__()
        self.hop_size = hop_size
        self.n_fft    = n_fft

    def forward(
        self,
        audio_mask: torch.Tensor,   # [B, L_audio], values 0 or 1
        time_max_length: int,       # T_spec (total time dim of the spectrogram)
    ) -> torch.Tensor:              # [B, T_spec], values 0.0 or 1.0

        B = audio_mask.shape[0]

        # Number of real (non-padded) audio samples per batch item
        audio_lengths = audio_mask.sum(dim=1).long()   # [B]

        # Apply the STFT formula: number of valid spectrogram frames
        spec_lengths = (
            (audio_lengths - self.n_fft) // self.hop_size + 1
        ).clamp(min=0)              # [B]

        # Build the binary mask
        spec_mask = torch.zeros(B, time_max_length, device=audio_mask.device)
        for i, valid_frames in enumerate(spec_lengths):
            spec_mask[i, :valid_frames] = 1.0

        return spec_mask

In [ ]:
# ── Quick sanity test ──────────────────────────────────────────────────
HOP, N_FFT = 512, 1024
audio_len   = 48_000         # 1 second at 48 kHz
pad_offset  = 8_000          # last 8000 samples are padding

audio_mask = torch.ones(1, audio_len)
audio_mask[0, audio_len - pad_offset:] = 0   # simulate padding at the end

real_len  = audio_len - pad_offset
T_spec    = (real_len - N_FFT) // HOP + 1
T_total   = (audio_len - N_FFT) // HOP + 1

layer = Audio2SpecPad(hop_size=HOP, n_fft=N_FFT)
spec_mask = layer(audio_mask, time_max_length=T_total)

print(f"Audio real length     : {real_len} samples")
print(f"Expected spec frames  : {T_spec}")
print(f"Mask active frames    : {int(spec_mask[0].sum())}")
assert int(spec_mask[0].sum()) == T_spec
print("✓ Audio2SpecPad test passed")

---
## TODO 3 — `Wav2Vec2FeatureExtractor` accepting batched `torch.Tensor` (1 point)

`Wav2Vec2FeatureExtractor` expects a list of numpy arrays. The wrapper below:
1. Accepts a padded `torch.Tensor` of shape `[B, L]` **and** its `padding_mask`.
2. Performs the same zero-mean / unit-variance normalization that the original extractor does.
3. Returns `input_values` ready to feed directly to the model — no numpy, no Python loops.

In [ ]:
class BatchedWav2Vec2FeatureExtractor:
    """
    Drop-in replacement for Wav2Vec2FeatureExtractor that works directly
    on a batched torch.Tensor instead of a list of numpy arrays.

    The original extractor normalises each waveform to zero mean / unit
    variance using only the *non-padded* region. We replicate that here
    entirely in PyTorch so the operation can run on GPU.

    Parameters
    ----------
    do_normalize : bool — if True, apply per-utterance mean/std normalization
                          (matches the default HF behaviour)
    """

    def __init__(self, do_normalize: bool = True):
        self.do_normalize = do_normalize

    def __call__(
        self,
        waveforms:     torch.Tensor,   # [B, L]  — zero-padded batch
        padding_masks: torch.Tensor,   # [B, L]  — 1 = real, 0 = padding
    ) -> torch.Tensor:                 # [B, L]  — normalised input_values

        waveforms = waveforms.float()
        masks     = padding_masks.float()

        if not self.do_normalize:
            return waveforms

        # Compute statistics only over the non-padded positions
        lengths = masks.sum(dim=1, keepdim=True).clamp(min=1)   # [B, 1]

        # Mean of real samples
        mean = (waveforms * masks).sum(dim=1, keepdim=True) / lengths  # [B, 1]

        # Std of real samples (population std, matching HF implementation)
        sq_diff = ((waveforms - mean) * masks) ** 2
        std  = (sq_diff.sum(dim=1, keepdim=True) / lengths).sqrt()     # [B, 1]
        std  = std.clamp(min=1e-7)

        # Normalise only the real positions; leave padding as zero
        normed = ((waveforms - mean) / std) * masks

        return normed


# ── Verify parity with the official HF extractor ──────────────────────
def verify_against_hf(n_samples=4, length=16_000):
    from transformers import Wav2Vec2FeatureExtractor as HFExtractor

    hf_extractor = HFExtractor.from_pretrained("facebook/wav2vec2-base-960h")
    our_extractor = BatchedWav2Vec2FeatureExtractor(do_normalize=True)

    # Random waves with different real lengths (simulate padding)
    real_lengths = torch.randint(8_000, length, (n_samples,))
    raw_waves    = torch.randn(n_samples, length)
    masks        = torch.zeros(n_samples, length)
    for i, l in enumerate(real_lengths):
        masks[i, :l]         = 1
        raw_waves[i, l:]     = 0   # zero out padding

    # HF reference: feed real parts only (as numpy lists)
    hf_inputs = hf_extractor(
        [raw_waves[i, :real_lengths[i]].numpy() for i in range(n_samples)],
        sampling_rate=16_000,
        padding=True,
        return_tensors="pt",
    ).input_values   # [B, length]

    # Our extractor
    our_inputs = our_extractor(raw_waves, masks)

    # Compare only the non-padded positions
    for i in range(n_samples):
        l = real_lengths[i]
        diff = (hf_inputs[i, :l] - our_inputs[i, :l]).abs().max().item()
        print(f"Sample {i}: max absolute diff = {diff:.2e}")
        assert diff < 1e-4, "Mismatch with HF reference!"

    print("✓ BatchedWav2Vec2FeatureExtractor matches HF reference.")


# verify_against_hf()  # uncomment when facebook/wav2vec2-base-960h is available

---
## TODO 4 — What does `Wav2Vec2ForCTC("facebook/wav2vec2-base-960h")` predict? (theory)

The output has shape **`[B, T, C]`** where `C = 32`.

Those 32 classes correspond to the **character-level CTC vocabulary**:

| token | meaning |
|-------|---------|
| `A`–`Z` (26 tokens) | uppercase English letters |
| `\|` | word boundary (space between words) |
| `[UNK]` | unknown character |
| `[PAD]` | CTC blank token (used to collapse repeated predictions) |
| `'` | apostrophe (contraction marker) |

The model applies **greedy CTC decoding**: at each time step take `argmax` over the 32 logits, then collapse consecutive identical tokens and remove all `[PAD]` tokens. The `|` boundary is converted to a space to reconstruct readable words.

You can inspect the full vocabulary with:

In [ ]:
from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("facebook/wav2vec2-base-960h")
# print(tokenizer.get_vocab())   # 32 entries
# => {'|': 4, "'": 5, 'A': 6, 'B': 7, ..., 'Z': 31,
#     '[UNK]': 0, '[PAD]': 1, '<s>': 2, '</s>': 3}

# Vocabulary summary (pre-computed, no model download needed)
vocab = (
    ["[PAD]", "[UNK]", "<s>", "</s>", "|", "'"] +
    [chr(c) for c in range(ord('A'), ord('Z') + 1)]
)
print(f"Vocabulary size: {len(vocab)}")
print("Tokens:", vocab)

---
## TODO 5 — Use `attention_mask` with `Wav2Vec2Model` (0.5 point)

Passing `attention_mask` tells the model which positions are real vs. padding.  
Benefits:
- The Transformer's self-attention ignores padded positions → **more accurate representations**.
- The returned `last_hidden_state` is still `[B, T_max, C]`, but padded frames are suppressed. This lets us unpad *without* the `Audio2SpecPad` heuristic: we can convert the `attention_mask` directly to the output time dimension using `Wav2Vec2Model._get_feat_extract_output_lengths()`.

In [ ]:
def extract_w2v_with_attention_mask(
    model,          # Wav2Vec2Model on device
    processor,      # AutoProcessor
    waveforms,      # list of 1-D np.ndarray  OR  [B, L] torch.Tensor
    sample_rate: int = 16_000,
    device: str = "cuda",
):
    """
    Run Wav2Vec2Model with proper attention_mask.

    Returns
    -------
    hidden_states : torch.Tensor [B, T_max, 768]
    output_mask   : torch.Tensor [B, T_max]  — 1 = valid frame, 0 = padding
    """
    # --- 1. Feature extraction & padding ---
    if isinstance(waveforms, torch.Tensor):
        wave_list = [waveforms[i].numpy() for i in range(len(waveforms))]
    else:
        wave_list = waveforms

    inputs = processor(
        wave_list,
        sampling_rate=sample_rate,
        return_tensors="pt",
        padding=True,            # pad all sequences to the longest in the batch
        return_attention_mask=True,   # <— this is the key flag
    )

    input_values  = inputs.input_values.to(device)   # [B, L]
    attention_mask = inputs.attention_mask.to(device) # [B, L]  1=real, 0=pad

    # --- 2. Forward pass WITH attention_mask ---
    with torch.no_grad():
        output = model(
            input_values,
            attention_mask=attention_mask,   # <— passed here
        )

    hidden_states = output.last_hidden_state   # [B, T_max, C]

    # --- 3. Derive output-level mask analytically ---
    # Wav2Vec2 CNN encoder reduces the sequence length by a fixed factor.
    # The model exposes a helper to compute the output lengths exactly:
    audio_lengths  = attention_mask.sum(dim=1)                     # [B]
    output_lengths = model._get_feat_extract_output_lengths(audio_lengths)  # [B]

    T_max       = hidden_states.shape[1]
    output_mask = torch.zeros(len(wave_list), T_max, device=device)
    for i, valid in enumerate(output_lengths):
        output_mask[i, :valid] = 1.0

    return hidden_states, output_mask


# Usage sketch (requires model + processor in scope):
# processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base-960h")
# model     = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to("cuda")
# hs, mask  = extract_w2v_with_attention_mask(model, processor, speaker_audio_10_16k)
# hs_valid  = hs[0][mask[0].bool()]   # unpadded frames for item 0

---
## TODO 6 — Normalize `HubertForCTC` outputs to W2V format (2 points)

**Root cause of the mismatch:** `processor.batch_decode` was called with a *1-D* token tensor (single sequence), so it treated each token id as a separate "batch item" and returned a list of individual characters instead of a single decoded string.

**Fix:** Call `processor.batch_decode` with a *2-D* tensor `[B, T]`, which triggers proper CTC decoding (collapse repeats → remove `<pad>` → join characters → restore spaces).

This produces the same `List[str]` format that `tokenizer.decode` gives for W2V.

In [ ]:
def decode_hubert_ctc(
    model_output,          # output of HubertForCTC.forward()
    processor,             # AutoProcessor for the same model
    skip_special_tokens: bool = True,
) -> list:
    """
    Decode HubertForCTC logits into a list of transcription strings,
    matching the format produced by Wav2Vec2ForCTC + tokenizer.decode().

    Parameters
    ----------
    model_output : CausalLMOutput — direct output of model(input_values)
    processor    : the AutoProcessor used to load the model
    
    Returns
    -------
    List[str]  — one transcribed string per batch item, e.g.
                 ["GUIDELINES", "FINALISED", ...]
    """
    # Step 1: greedy argmax over vocabulary dimension
    #   logits shape: [B, T, vocab_size]
    #   pred_ids shape: [B, T]
    pred_ids = torch.argmax(model_output.logits, dim=-1)   # [B, T]

    # Step 2: CTC decoding — processor.batch_decode handles:
    #   • collapsing consecutive identical tokens
    #   • removing <pad> (blank) tokens
    #   • joining remaining characters into a string
    #   • converting the '|' word-boundary token to a space
    #
    # KEY: pass the FULL [B, T] tensor, NOT a slice per sample.
    transcriptions = processor.batch_decode(
        pred_ids,
        skip_special_tokens=skip_special_tokens,
    )

    return transcriptions   # List[str], length = B


# ── Example (requires model in scope) ─────────────────────────────────
# processor = AutoProcessor.from_pretrained("facebook/hubert-large-ls960-ft")
# model     = HubertForCTC.from_pretrained("facebook/hubert-large-ls960-ft").cuda()
#
# inputs     = processor([...waves...], sampling_rate=16_000, return_tensors="pt")
# with torch.no_grad():
#     out = model(inputs.input_values.cuda())
#
# transcriptions = decode_hubert_ctc(out, processor)
# print(transcriptions)
# => ["GUIDELINES", "FINALISED", ...]   ← same format as W2V

# ── Verify the OLD (broken) approach vs the fix ────────────────────────
print("OLD approach — processor.batch_decode called on 1-D slice [T]:")
print("  Returns a list of individual tokens: ['G','U','I','D','E',...]")
print("  You'd then need: ' '.join([t for t in transcription if t != '<pad>'])")
print()
print("NEW approach — processor.batch_decode called on 2-D tensor [B, T]:")
print("  Returns proper strings: ['GUIDELINES', 'FINALISED', ...]")
print("  Identical in format to Wav2Vec2 tokenizer.decode() output.")

---
## TODO 7a — Add batching to `XVectorExtractor` (1 point)

In [ ]:
class XVectorExtractor:
    """
    Batched x-vector extractor using AutoModelForAudioXVector.

    Changes from the original:
    - processes `batch_size` files at a time instead of one-by-one
    - uses the feature_extractor's built-in padding so all waveforms in
      a batch are padded to the same length automatically
    - feeds the resulting `attention_mask` to the model for accuracy
    """

    def __init__(
        self,
        model_name: str,
        device: str     = "cpu",
        do_normalize: bool = True,
        batch_size: int = 8,          # <— new: controls how many files per forward pass
    ):
        self.feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
        self.model             = AutoModelForAudioXVector.from_pretrained(model_name).to(device)
        self.do_normalize      = do_normalize
        self.device            = device
        self.batch_size        = batch_size

    def extract_features(
        self,
        wave_paths: list,
        sample_rate: int,
    ) -> torch.Tensor:                # [N_files, embedding_dim]

        all_embeddings = []

        # ── Iterate in chunks of batch_size ──────────────────────────
        for start in tqdm(range(0, len(wave_paths), self.batch_size)):
            batch_paths = wave_paths[start : start + self.batch_size]

            # Load all waveforms in the batch
            waves = [
                librosa.load(p, sr=sample_rate)[0]
                for p in batch_paths
            ]

            # Feature extraction with padding so all waves → same length
            inputs = self.feature_extractor(
                waves,
                sampling_rate=sample_rate,
                return_tensors="pt",
                padding=True,               # <— pads to longest in batch
                return_attention_mask=True, # <— proper mask for the model
            )

            input_values  = inputs.input_values.to(self.device)
            attention_mask = inputs.attention_mask.to(self.device)

            # Forward pass
            with torch.no_grad():
                embeddings = self.model(
                    input_values,
                    attention_mask=attention_mask,
                ).embeddings                # [batch_size, emb_dim]

            if self.do_normalize:
                embeddings = F.normalize(embeddings, dim=-1)

            all_embeddings.append(embeddings.cpu())

        return torch.cat(all_embeddings, dim=0)   # [N_files, emb_dim]

---
## TODO 7b — How to transform `Wav2Vec2Model` into `Wav2Vec2ForXVector`? (0.5 point)

`Wav2Vec2ForXVector` is `Wav2Vec2Model` + a small speaker-classification head. The architecture adds three components on top of the shared encoder:

```
Wav2Vec2Model
  └── wav2vec2  (CNN feature extractor + Transformer encoder)
       │
       ▼  [B, T, 768]
  projector  (Linear 768 → 512, no bias)
       │
       ▼  [B, T, 512]
  tdnn  (5 × TDNN / 1-D Conv blocks with dilation)
       │
       ▼  [B, T', 1500]
  feature_extractor  (mean + std pooling → 3000-dim stats vector)
       │
       ▼  [B, 3000]
  classifier  (Linear 3000 → 256  → Linear 256 → n_speakers)
       │
       ▼  embeddings [B, 256]
```

In code:

In [ ]:
import torch.nn as nn
from transformers import Wav2Vec2Model
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2ForXVector,
    Wav2Vec2FeatureProjection,
)


class TDNNBlock(nn.Module):
    """Single Time-Delay Neural Network block (1-D dilated conv + ReLU)."""
    def __init__(self, in_dim, out_dim, kernel_size, dilation):
        super().__init__()
        self.conv = nn.Conv1d(
            in_dim, out_dim, kernel_size,
            dilation=dilation,
            padding=dilation * (kernel_size - 1) // 2,
        )
        self.relu = nn.ReLU()

    def forward(self, x):          # x: [B, C, T]
        return self.relu(self.conv(x))


class Wav2Vec2AsXVector(nn.Module):
    """
    Wraps a pre-trained Wav2Vec2Model and adds the x-vector head on top.
    The base encoder weights are frozen by default (fine-tune head only).

    Architecture mirrors HuggingFace's Wav2Vec2ForXVector:
      wav2vec2  →  projector  →  TDNN stack  →  stats pooling  →  embedding.
    """

    def __init__(
        self,
        wav2vec2_name: str = "facebook/wav2vec2-base",
        hidden_size:   int = 768,    # match wav2vec2 hidden dim
        proj_dim:      int = 512,
        emb_dim:       int = 256,
        n_speakers:    int = 512,
        freeze_encoder: bool = True,
    ):
        super().__init__()

        # ── Shared SSL encoder ────────────────────────────────────────
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(wav2vec2_name)

        if freeze_encoder:
            for p in self.wav2vec2.parameters():
                p.requires_grad = False

        # ── Speaker-discrimination head ───────────────────────────────
        # 1. Projection: reduce hidden dim
        self.projector = nn.Linear(hidden_size, proj_dim, bias=False)

        # 2. TDNN stack (matches default HF config)
        tdnn_cfg = [
            # (in_dim, out_dim, kernel, dilation)
            (proj_dim, 512, 5, 1),
            (512,       512, 3, 2),
            (512,       512, 3, 3),
            (512,       512, 1, 1),
            (512,      1500, 1, 1),
        ]
        self.tdnn = nn.Sequential(*[
            TDNNBlock(*cfg) for cfg in tdnn_cfg
        ])

        # 3. Stats pooling: mean + std → 3000-dim vector
        #    (implemented in forward)

        # 4. Embedding / classifier layers
        self.fc1        = nn.Linear(3000, emb_dim)
        self.classifier = nn.Linear(emb_dim, n_speakers)

    def forward(
        self,
        input_values:  torch.Tensor,              # [B, L]
        attention_mask: torch.Tensor | None = None,
    ):
        # Step 1 — SSL encoder
        hidden = self.wav2vec2(
            input_values, attention_mask=attention_mask
        ).last_hidden_state                        # [B, T, 768]

        # Step 2 — Project
        x = self.projector(hidden)                 # [B, T, proj_dim]

        # Step 3 — TDNN (expects [B, C, T])
        x = x.transpose(1, 2)                      # [B, proj_dim, T]
        x = self.tdnn(x)                           # [B, 1500, T']

        # Step 4 — Stats pooling: concatenate mean and std over time
        mean = x.mean(dim=-1)                      # [B, 1500]
        std  = x.std(dim=-1)                       # [B, 1500]
        stats = torch.cat([mean, std], dim=-1)     # [B, 3000]

        # Step 5 — x-vector embedding
        embeddings = F.relu(self.fc1(stats))       # [B, emb_dim]

        # Step 6 — Speaker logits
        logits = self.classifier(embeddings)       # [B, n_speakers]

        return embeddings, logits


# Confirm architecture compiles
# model = Wav2Vec2AsXVector(n_speakers=100)
# dummy_input = torch.randn(2, 16_000)
# emb, logits = model(dummy_input)
# print("embeddings:", emb.shape)   # [2, 256]
# print("logits:    ", logits.shape) # [2, 100]

---
## Summary

| # | What was done |
|---|---------------|
| 1 | `UniversalSlicer` with 5 policies: `right_pad`, `left_pad`, `center_pad`, `cut`, `cut_silence` |
| 2 | `Audio2SpecPad` rewritten using the STFT length formula instead of `MaxPool + manual cat` |
| 3 | `BatchedWav2Vec2FeatureExtractor` normalises a padded `torch.Tensor [B, L]` on GPU, results verified against HF reference |
| 4 | `Wav2Vec2ForCTC` predicts 32 CTC tokens: A–Z, `|` (space), `'`, `[PAD]`, `[UNK]`, `<s>`, `</s>` |
| 5 | `extract_w2v_with_attention_mask()` passes `attention_mask` to `Wav2Vec2Model` and uses `_get_feat_extract_output_lengths()` for an exact output mask |
| 6 | `decode_hubert_ctc()` calls `processor.batch_decode` on the full `[B, T]` tensor → same `List[str]` format as W2V |
| 7 | Batched `XVectorExtractor` + `Wav2Vec2AsXVector` showing the exact architectural transformation from base SSL encoder to x-vector model |